# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c23 — Design Space Optimizer  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Optimise structural geometry using CMA-ES that jointly minimises material cost and maximises reclaimed-timber reuse rate. Each candidate design passes through a full pipeline: geometry → feasibility (mean-EA) → cost matrix → MILP assignment → GNN structural check → fitness score.

### Inputs
- `search_space_{GRID}.json` — discrete/continuous design variables (from c12_15)
- `complete_timber_A.csv` / `complete_timber_B.csv` — available timber stock (set in cell 1)
- Trained GNN surrogate model (prefix set in cell 1)

### Outputs
- Optimised design candidates with fitness scores
- Ranked solution set and run metadata exported to `config.RESULTS_PATH`

### Notebook flow
| Section | Purpose |
|---------|---------|
| 1. Setup | Load stock, search space, and configure the GA — run once per session |
| 2. Evaluation | Geometry check + one-time normalisation bounds solve |
| 3. GA Run | Execute CMA-ES |
| 4. Analysis | Inspect convergence and solution quality |
| 5. Export | Save candidates and run metadata to disk |

# 1. Setup
Load all dependencies, stock data, and search space. Configure GA weights, MILP constraints, and the surrogate model prefix. **Run once per session** before the evaluation or GA cells.

In [1]:
# Run once per notebook session before the GA loop.
# Loads all dependencies, stock data, search space, and GA configuration.
#
# Produces the following session variables:
#   optimizer_search_space        — dict[str, (float, float)]
#   df_input_stock                — pd.DataFrame
#   GA_CONFIG                     — pipeline-level settings dict
#   FIXED_NORMALIZATION_CONSTANTS — initial normalisation bounds
#   MODEL_PREFIX                  — surrogate model artifact stem
#   PREPARED_GNN_STOCK            — SI-unit stock for GNN (computed once here)

import importlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import config
import c00_headquarter_params as c11_params
from workflows import c22_stage_geometry             as stage_geometry
from workflows import c24_stage_feasibility          as stage_feas
from workflows import c25_stage_cost_matrix          as stage_cost
from workflows import c26_stage_MILP                 as stage_milp
from workflows import c27_stage_GNN                  as stage_gnn
from workflows import c28_stage_fitness_score        as stage_fitness
from workflows import c28_stage_normalization_bounds as stage_bounds
from workflows import c23_ga_evaluator               as ga_eval

# Hot-reload all workflow modules so notebook edits are picked up immediately
for _mod in [stage_geometry, stage_feas, stage_cost,
             stage_milp, stage_gnn, stage_fitness, stage_bounds]:
    importlib.reload(_mod)
importlib.reload(ga_eval)

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# FLAGS
# =============================================================================

TESTING = False   # True → loads small stock CSV for fast development runs

# =============================================================================
# SEARCH SPACE
# =============================================================================

json_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
if not json_path.exists():
    raise FileNotFoundError(
        f"search_space_{c11_params.GRID}.json not found at {json_path}. "
        "Generate it from the geometry notebook (c12_15) before running the GA."
    )
with open(json_path, "r", encoding="utf-8") as f:
    optimizer_search_space = json.load(f)

print(f"Search space loaded: {len(optimizer_search_space)} variables")

# =============================================================================
# STOCK DATA
# =============================================================================

stock_file = (
    config.TIMBER_STOCK_PATH / "complete_timber_small.csv"
    if TESTING
    else config.TIMBER_STOCK_PATH / "complete_timber_A.csv"
)
if not stock_file.exists():
    raise FileNotFoundError(f"Stock CSV not found: {stock_file}")

_df_input_stock = None
for _opts in [
    {"sep": ",",  "encoding": "utf-8"},
    {"sep": ";",  "encoding": "utf-8"},
    {"sep": ",",  "encoding": "latin1"},
    {"sep": ";",  "encoding": "latin1"},
]:
    try:
        _df = pd.read_csv(stock_file, **_opts)
        if _df.shape[1] > 1:
            _df_input_stock = _df
            print(f"Stock loaded: {len(_df)} elements  "
                  f"(sep='{_opts['sep']}', encoding='{_opts['encoding']}')")
            break
    except Exception:
        pass

if _df_input_stock is None:
    raise ValueError(
        f"Could not parse stock CSV at {stock_file}. "
        "Tried , and ; delimiters with utf-8 and latin1 encoding."
    )

df_input_stock = _df_input_stock
df_input_stock.columns = df_input_stock.columns.str.strip()

PREPARED_GNN_STOCK = stage_gnn.prepare_stock_for_gnn(df_input_stock)
print(f"Stock loaded from {stock_file}")
print(f"GNN stock prepared: {len(PREPARED_GNN_STOCK)} elements\n"
      f"(Width_m, Depth_m, E, Iy, Iz, J added)")

# =============================================================================
# SURROGATE MODEL PREFIX
# Trained 2026-05-16 on v6 dataset (9 edge features: Width_m, Depth_m, Length,
# E, Iy, Iz, J, EA/L, N_mean_EA). ROC AUC=0.860, best epoch=200.
# hidden_dim=64, num_layers=3, dropout=0.3, weight_decay=1e-3, lr=1e-4.
# =============================================================================

MODEL_PREFIX = "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"

# =============================================================================
# GA CONFIGURATION
#
# Pipeline-level settings only — weights, MILP, penalty, bounds.
# ES algorithm parameters live in ESConfig in c23_ga_algorithm.py.
# =============================================================================

GA_CONFIG = {
    # Fitness weights — omega_1: cost, omega_2: reuse rate
    "fitness_weights": {
        "omega_1": 1.0,
        "omega_2": 3.0,
    },

    # MILP — how many times a new-stock element may be assigned across slots
    "new_stock_max_uses": 10,

    # Minimum fraction of slots that must be assigned a reclaimed (RS) element.
    # Run 4 confirmed the stock pool feasibility cap sits at ~21%, so 0.20
    # matches the achievable ceiling without pushing MILP beyond what the stock allows.
    # Set to None (or 0.0) to disable.
    "min_reuse_fraction": 0.0,

    # Penalty fitness assigned to infeasible or failed pipeline evaluations
    "penalty_fitness": 1e6,

    # One-time normalisation bounds computed once at GA startup
    "use_one_time_bounds":   True,
    "bounds_probe_attempts": 8,

    # Structural penalty curriculum — omega_4 fixed at 0.8 from gen 0
    "w_structural_start": 0.8,
    "w_structural_end":   0.8,
}

# =============================================================================
# INITIAL NORMALISATION CONSTANTS
# Placeholder — overwritten by the one-time bounds solve in cell 2.2
# =============================================================================

FIXED_NORMALIZATION_CONSTANTS = stage_fitness.get_default_normalization_constants()

# =============================================================================
# SUMMARY
# =============================================================================

_mrf = GA_CONFIG.get("min_reuse_fraction")
_mrf_str = f"{_mrf:.0%}" if _mrf else "None (unconstrained)"

print(f"\n{'='*55}")
print("GA SESSION READY")
print(f"{'='*55}")
print(f"  Search space variables:  {len(optimizer_search_space)}")
print(f"  Stock elements:          {len(df_input_stock)}")
print(f"  Model prefix:            {MODEL_PREFIX}")
print(f"  Fitness weights:         "
      f"ω1={GA_CONFIG['fitness_weights']['omega_1']}  "
      f"ω2={GA_CONFIG['fitness_weights']['omega_2']}")
print(f"  Structural schedule:     "
      f"ω4 {GA_CONFIG['w_structural_start']} → {GA_CONFIG['w_structural_end']}")
print(f"  New stock max uses:      {GA_CONFIG['new_stock_max_uses']}")
print(f"  Min reuse fraction:      {_mrf_str}")
print(f"  Fixed normalization:     {FIXED_NORMALIZATION_CONSTANTS}")
print(f"  Penalty fitness:         {GA_CONFIG['penalty_fitness']:.0e}")
print(f"  TESTING mode:            {TESTING}")
print(f"{'='*55}\n")

display(df_input_stock.head(3))

Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

Grid: 5x3, edge=3.0 m, height=1.5 m, divisions=8, samples=20000
LCA factors: A1-A3=0.25, C1=0.0085, A5 prep=0.01, A5 saw=0.01, C2 dist=50 km, C3-C4=0.276, ω=0


c:\Users\jaspe\Documents\PyEnvs\thesis_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Search space loaded: 73 variables
Stock loaded: 522 elements  (sep=';', encoding='utf-8')
Stock loaded from C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\03_timber_data\complete_timber_A.csv
GNN stock prepared: 522 elements
(Width_m, Depth_m, E, Iy, Iz, J added)

GA SESSION READY
  Search space variables:  73
  Stock elements:          522
  Model prefix:            ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863
  Fitness weights:         ω1=1.0  ω2=3.0
  Structural schedule:     ω4 0.8 → 0.8
  New stock max uses:      10
  Min reuse fraction:      None (unconstrained)
  Fixed normalization:     {'C_max': 8.0, 'R_max': 1.0}
  Penalty fitness:         1e+06
  TESTING mode:            False



,Member_ID,State,Length,Depth,Width,Length_Category,Availability_Probability,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor,Donor_Role,Cut_Loss_mm
0,NS_00000,0,1800.0,100.0,38.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,323.62,0.1747,NaN,NaN
1,NS_00001,0,1800.0,100.0,50.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Netherlands,48.35,0.1753,NaN,NaN
2,NS_00002,0,1800.0,100.0,63.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,256.40,0.1775,NaN,NaN


# 2. Evaluation pipeline
Pre-checks before the GA run. Verify geometry topology and solve for one-time normalisation constants. Both cells should complete without errors before starting section 3.

## 2.1 Geometry check
Generates one random structure and verifies the expected node and member count. Set `RUN_GEOMETRY_CHECK = False` to skip during production runs.

In [2]:
# =============================================================================
# c30_ga_geometry_check.py — One-time Geometry Sanity Check
# =============================================================================
#
# Optional: run once before the GA to verify the geometry stage produces
# the expected topology (39 nodes, 120 members) and that the output columns
# match what downstream stages expect.
#
# Set RUN_GEOMETRY_CHECK = True to execute; False to skip silently.
# Keep False during production GA runs — this adds ~5-30s unnecessarily.

RUN_GEOMETRY_CHECK = True

EXPECTED_NODES   = 39
EXPECTED_MEMBERS = 120

if RUN_GEOMETRY_CHECK:
    geometry_out = stage_geometry.run_random_geometry_stage(
        optimizer_search_space=optimizer_search_space,
        sample_id=0,
    )

    df_vertices_example         = geometry_out["df_vertices"]
    df_edges_example            = geometry_out["df_edges"]
    df_geometry_overview_example = geometry_out["df_geometry_overview"]

    n_nodes   = len(df_vertices_example)
    n_members = len(df_edges_example)

    print(f"Geometry check: {n_nodes} nodes, {n_members} members")

    if n_nodes != EXPECTED_NODES:
        print(f"  WARNING: expected {EXPECTED_NODES} nodes, got {n_nodes}")
    else:
        print(f"  Nodes:   {n_nodes} / {EXPECTED_NODES} ✓")

    if n_members != EXPECTED_MEMBERS:
        print(f"  WARNING: expected {EXPECTED_MEMBERS} members, got {n_members}")
    else:
        print(f"  Members: {n_members} / {EXPECTED_MEMBERS} ✓")

    display(df_geometry_overview_example[["edge_id", "V1", "V2", "length_m"]].head(5))

else:
    print("Geometry check skipped (RUN_GEOMETRY_CHECK = False).")

Geometry check: 39 nodes, 120 members
  Nodes:   39 / 39 ✓
  Members: 120 / 120 ✓


,edge_id,V1,V2,length_m
0,e0,0,1,3.750
1,e1,0,6,2.625
2,e2,1,2,1.125
3,e3,1,7,2.704
4,e4,2,3,3.000


## 2.2 Bounds solve + smoke test
Probes the full pipeline to compute one-time normalisation constants, then runs a single end-to-end evaluation (geometry → feasibility → MILP → GNN → fitness) to confirm all stages are wired correctly before launching the GA.

In [3]:
# =============================================================================
# ONE-TIME BOUNDS SOLVE + SMOKE TEST
# =============================================================================

FIXED_NORMALIZATION_CONSTANTS, BOUNDS_SOURCE_INFO = (
    ga_eval._compute_one_time_normalization_constants(
        search_space = optimizer_search_space,
        df_stock     = df_input_stock,
        config_dict  = GA_CONFIG,
    )
)
print(f"\nNormalisation constants: {FIXED_NORMALIZATION_CONSTANTS}")
print(f"Source: {BOUNDS_SOURCE_INFO}")

print("\nRunning evaluator smoke test...")
_test_design = stage_geometry.sample_random_design(optimizer_search_space)
_test_eval   = ga_eval.evaluate_design_candidate(
    design_params        = _test_design,
    df_stock             = df_input_stock,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    config_dict          = GA_CONFIG,
    model_prefix         = MODEL_PREFIX,
    generation           = 0,
    max_generations      = 1,
    sample_id            = 99_999,
    verbose              = True,
    prepared_gnn_stock   = PREPARED_GNN_STOCK,
)
print(f"Smoke test status: {_test_eval['status']}")
if _test_eval["reason"] is not None:
    print(f"Reason: {_test_eval['reason']}")
print("Evaluator ready.")

  Stage 1 (length):    43,224 eliminated  (19,416 remaining, 31.0%)
  Stage 2 (force):   max tension=13.9 kN  max compression=-19.2 kN  mean |F|=4.4 kN
  Stage 3 (EC5):        1,692 eliminated  (17,724 remaining, 28.3%)
  [bounds probe 1] success → {'C_max': 332.3151362899342, 'R_max': 0.6635166650754704}

Normalisation constants: {'C_max': 332.3151362899342, 'R_max': 0.6635166650754704}
Source: {'source': 'one-time-bounds', 'status': 'Optimal', 'attempt': 1}

Running evaluator smoke test...
    ✓ geometry    | 39 nodes, 120 edges
  Stage 1 (length):    42,812 eliminated  (19,828 remaining, 31.7%)
  Stage 2 (force):   max tension=26.4 kN  max compression=-20.4 kN  mean |F|=5.5 kN
  Stage 3 (EC5):        1,701 eliminated  (18,127 remaining, 28.9%)
    ✓ feasibility | 18,127 feasible slot/stock pairs
    ✓ cost matrix | 18,127 finite entries
    ✓ MILP        | status=Optimal, cost=142.7054, 120 assignments
[IO] Inference config loaded from ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0

# 3. GA run
Loads the surrogate bundle, wires the evaluation function, and runs CMA-ES. Tune `CMAESConfig` parameters (`popsize`, `n_generations`, `sigma_init`) to adjust exploration budget.

In [ ]:
import time
import c23_ga_algorithm as ga_algo
from c21_surrogate_io import load_surrogate_bundle

importlib.reload(ga_algo)
importlib.reload(ga_eval)

SURROGATE_BUNDLE = load_surrogate_bundle(prefix_sm=MODEL_PREFIX)
print(f"Bundle pre-loaded: {MODEL_PREFIX}")

evaluate_fn = ga_algo.make_evaluate_fn(
    evaluate_fn_raw      = ga_eval.evaluate_design_candidate,
    df_stock             = df_input_stock,
    fixed_norm_constants = FIXED_NORMALIZATION_CONSTANTS,
    config_dict          = GA_CONFIG,
    bundle               = SURROGATE_BUNDLE,
    prepared_gnn_stock   = PREPARED_GNN_STOCK,
    verbose              = True,
)

# ---- timing wrapper ----
_base_fn = evaluate_fn
def evaluate_fn(params, generation=0, max_generations=1):
    print()
    t0 = time.time()
    fitness, res = _base_fn(params, generation, max_generations)
    elapsed = time.time() - t0
    if res:
        status  = res.get("status",        "?")
        milp    = res.get("milp_status",    "?")
        gnn     = res.get("gnn_feasibility")
        cost    = res.get("total_cost",     float("nan"))
        reuse   = res.get("reuse_fraction", float("nan"))
        w4      = res.get("w_structural",   float("nan"))
        gnn_str = f"{gnn:.3f}" if gnn is not None else "skip"
        print(
            f"  → {elapsed:.1f}s | {status} | MILP={milp} | "
            f"GNN={gnn_str} | cost={cost:.1f} | reuse={reuse:.3f} | "
            f"ω4={w4:.2f} | fit={fitness:.4f}"
        )
    else:
        print(f"  → {elapsed:.1f}s | fitness={fitness:.4f}  (no result dict)")
    return fitness, res
# ------------------------


def _ss_to_bounds(ss):
    """Convert JSON search space entries to (lo, hi) tuples."""
    out = {}
    for k, v in ss.items():
        if v["type"] == "discrete":
            out[k] = (float(min(v["options"])), float(max(v["options"])))
        else:
            out[k] = (float(v["min"]), float(v["max"]))
    return out

es_search_space = _ss_to_bounds(optimizer_search_space)

# -----------------------------------------------------------------------------
# CMA-ES configuration
# -----------------------------------------------------------------------------
# Budget: popsize × n_generations = 30 × 250 = 7,500 evaluations ≈ 2–2.5 hours.
# CMA-ES may stop earlier if it converges (tolx/tolfun). Stop reasons are
# printed in the summary at the end of the run.
#
# Key differences from the previous (μ+λ)-ES:
#   - CMA-ES adapts the full covariance matrix, not just per-parameter σ values.
#     This lets it discover correlations between node positions automatically.
#   - Parameters are normalised to [0,1]^n internally; evaluate_fn receives
#     physical units unchanged.
#   - stagnation_limit is used only for the reference line in the analysis plot;
#     CMA-ES manages its own convergence via tolx/tolfun.

es = ga_algo.CMAEvolutionStrategy(
    search_space = es_search_space,
    evaluate_fn  = evaluate_fn,
    config       = ga_algo.CMAESConfig(
        popsize          = 30,
        n_generations    = 250,
        sigma_init       = 0.25,
        sigma_min        = 1e-8,
        stagnation_limit = 30,
        log_every        = 5,
    ),
    seed = 42,
)

result = es.run()


[IO] Inference config loaded from ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863_inference_config.json
[IO] Checkpoint loaded — epoch 186  val_loss=0.589257
[TrussEdgeSafetyGNN] Topology cached: 120 edges
[IO] Model ready on cpu  |  edge_index: (2, 120)  |  unidirectional
Bundle pre-loaded: ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863
[CMA-ES] Initialised: popsize(lam)=30  n_generations=250  n_params=73
         sigma_init=0.25  Expected evaluations: 7500

    ✓ geometry    | 39 nodes, 120 edges
  Stage 1 (length):    43,274 eliminated  (19,366 remaining, 30.9%)
  Stage 2 (force):   max tension=12.6 kN  max compression=-16.8 kN  mean |F|=4.7 kN
  Stage 3 (EC5):        1,602 eliminated  (17,764 remaining, 28.4%)
    ✓ feasibility | 17,764 feasible slot/stock pairs
    ✓ cost matrix | 17,764 finite entries
    ✓ MILP        | status=Optimal, cost=141.7938, 120 assignments
    ✓ GNN         | feasibility=57.73%, unsafe=72 members
    ✓ fitness     | fitness=-0.3107, cost=141.7

# 4. Analysis
Inspect GA convergence, fitness trajectory, and the quality of the best-found solutions across the run.

In [ ]:
from workflows import c23_ga_analysis_export as ga_ae
importlib.reload(ga_ae)

analysis_out = ga_ae.run_analysis(
    result                 = result,
    fixed_norm_constants   = FIXED_NORMALIZATION_CONSTANTS,
    optimizer_search_space = optimizer_search_space,
    stagnation_limit       = es.config.stagnation_limit,
)

# 5. Export
Save optimised design candidates, run configuration, and GA metadata to disk for downstream use in Grasshopper and reporting.

In [ ]:
importlib.reload(ga_ae)

export_out = ga_ae.run_export(
    analysis_out        = analysis_out,
    result              = result,
    ga_config           = GA_CONFIG,
    fixed_norm_constants= FIXED_NORMALIZATION_CONSTANTS,
    model_prefix        = MODEL_PREFIX,
    bounds_source_info  = BOUNDS_SOURCE_INFO,
    es                  = es,
)
print("Saved to:", export_out["export_dir"])